# Module 1
### Sensor Simulation (Python) 

In [6]:
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta


# ── CONSTANTS ────────────────────────────────────────────────────────────────

NORMAL_HR      = 100   # bpm  (dog resting HR: 60-140, active: up to 180)
NORMAL_TEMP    = 38.5  # °C   (dog normal: 38.0–39.2°C)
BASE_LAT       = 33.80
BASE_LON       = 151.20

# Thresholds for flagging (used later by anomaly module too)
HR_HIGH        = 160
HR_LOW         = 50
TEMP_HIGH      = 39.5
TEMP_LOW       = 37.5


# ── 1. DATA SIMULATION ───────────────────────────────────────────────────────

def generate_sensor_stream(dog_id: str, n_samples: int = 100,
                            inject_anomaly: bool = True) -> pd.DataFrame:
    """
    Simulates `n_samples` seconds of collar sensor readings.

    HOW IT WORKS:
      - Gaussian noise around a baseline models real sensor jitter
      - An optional stress event is injected mid-stream (HR + temp spike)
      - GPS drifts slightly each step (random walk)

    Returns a DataFrame with columns:
      timestamp, dog_id, heart_rate, temperature, lat, lon
    """
    np.random.seed(42)

    timestamps = [datetime(2026, 3, 10, 21, 0, 0) + timedelta(seconds=i)
                  for i in range(n_samples)]

    # Base signals: normal + gaussian noise
    hr   = np.random.normal(loc=NORMAL_HR,   scale=5,   size=n_samples)
    temp = np.random.normal(loc=NORMAL_TEMP, scale=0.2, size=n_samples)

    # GPS random walk (small drift per step)
    lat  = BASE_LAT + np.cumsum(np.random.normal(0, 0.0001, n_samples))
    lon  = BASE_LON + np.cumsum(np.random.normal(0, 0.0001, n_samples))

    # Inject a stress anomaly in the middle third of the stream
    if inject_anomaly:
        anomaly_start = n_samples // 3
        anomaly_end   = anomaly_start + 20
        hr[anomaly_start:anomaly_end]   += np.linspace(0, 65, 20)   # ramp up HR
        temp[anomaly_start:anomaly_end] += np.linspace(0, 1.8, 20)  # ramp up temp

    df = pd.DataFrame({
        "timestamp":  timestamps,
        "dog_id":     dog_id,
        "heart_rate": np.round(hr,   1),
        "temperature":np.round(temp, 2),
        "lat":        np.round(lat,  5),
        "lon":        np.round(lon,  5),
    })
    return df


# ── 2. PREPROCESSING ─────────────────────────────────────────────────────────

def preprocess(df: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    """
    Applies two preprocessing steps to the raw sensor stream.

    STEP A — Rolling Mean (smoothing):
      Computes a sliding window average to reduce momentary noise spikes.
      Window=5 means each value is the average of itself + 4 prior readings.
      This is essentially a low-pass filter in signal processing.

      WHY: A single noisy spike shouldn't trigger an alert.
           We want sustained elevation to count.

    STEP B — Z-Score Normalization:
      z = (x - mean) / std
      Transforms each reading into "how many standard deviations from normal".
      z > 2.0  →  unusually high
      z < -2.0 →  unusually low

      WHY: Makes HR and temperature comparable on the same scale,
           which matters for the anomaly detection module.
    """
    df = df.copy()

    # A: Rolling smoothed versions
    df["hr_smooth"]   = df["heart_rate"].rolling(window=window, min_periods=1).mean()
    df["temp_smooth"] = df["temperature"].rolling(window=window, min_periods=1).mean()

    # B: Z-scores on the smoothed signals
    df["hr_zscore"]   = (df["hr_smooth"]   - df["hr_smooth"].mean())   / df["hr_smooth"].std()
    df["temp_zscore"] = (df["temp_smooth"] - df["temp_smooth"].mean()) / df["temp_smooth"].std()

    df["hr_anomaly_score"] = abs(df["hr_zscore"])
    df["temp_anomaly_score"] = abs(df["temp_zscore"])
    
    return df


# ── 3. EVENT OBJECT BUILDER ──────────────────────────────────────────────────

def build_event(row: pd.Series) -> dict:
    flags = []

    # Absolute thresholds (same as before)
    if row["hr_smooth"] > HR_HIGH:
        flags.append("high_hr")
    if row["hr_smooth"] < HR_LOW:
        flags.append("low_hr")
    if row["temp_smooth"] > TEMP_HIGH:
        flags.append("high_temp")
    if row["temp_smooth"] < TEMP_LOW:
        flags.append("low_temp")

    # NEW: relative anomaly (matches Module 2 thinking)
    if abs(row["hr_zscore"]) > 2:
        flags.append("hr_outlier")

    if abs(row["temp_zscore"]) > 2:
        flags.append("temp_outlier")

    return {
        "dog_id": row["dog_id"],
        "timestamp": row["timestamp"].isoformat(),
        "heart_rate": round(row["hr_smooth"], 1),
        "temperature": round(row["temp_smooth"], 2),
        "gps": {"lat": row["lat"], "lon": row["lon"]},
        "flags": flags,
    }


# ── MAIN ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Step 1: Generate raw stream
    raw_df = generate_sensor_stream(dog_id="Rex", n_samples=100, inject_anomaly=True)
    print("=== Raw Stream (first 5 rows) ===")
    print(raw_df.head())

    # Step 2: Preprocess
    proc_df = preprocess(raw_df, window=5)
    print("\n=== Preprocessed (with z-scores, first 5 rows) ===")
    print(proc_df[["timestamp", "heart_rate", "hr_smooth", "hr_zscore",
                   "temperature", "temp_smooth", "temp_zscore"]].head())

    # Step 3: Find anomalous rows and build event objects
    events = []
    for _, row in proc_df.iterrows():
        event = build_event(row)
        if event["flags"]:
            events.append(event)

    print(f"\n=== {len(events)} Anomalous Events Detected ===")
    if events:
        print("First flagged event:")
        print(json.dumps(events[0], indent=2))

    # Save
    import os
    os.makedirs("data", exist_ok=True)
    proc_df.to_csv("data/sensor_data_processed.csv", index=False)
    with open("data/anomaly_events.json", "w") as f:
        json.dump(events, f, indent=2)

    print("\nSaved: data/sensor_data_processed.csv, data/anomaly_events.json")

    # Save preprocessed data and events for next modules
    proc_df.to_csv("data/sensor_data_processed.csv", index=False)
    with open("data/anomaly_events.json", "w") as f:
        json.dump(events, f, indent=2)

    print("\nSaved: sensor_data_processed.csv, anomaly_events.json")
    print("→ These files feed into Module 2 (Anomaly Detection)")

=== Raw Stream (first 5 rows) ===
            timestamp dog_id  heart_rate  temperature       lat        lon
0 2026-03-10 21:00:00    Rex       102.5        38.22  33.80004  151.19992
1 2026-03-10 21:00:01    Rex        99.3        38.42  33.80009  151.19986
2 2026-03-10 21:00:02    Rex       103.2        38.43  33.80020  151.19994
3 2026-03-10 21:00:03    Rex       107.6        38.34  33.80031  151.20000
4 2026-03-10 21:00:04    Rex        98.8        38.47  33.80017  151.19999

=== Preprocessed (with z-scores, first 5 rows) ===
            timestamp  heart_rate   hr_smooth  hr_zscore  temperature  \
0 2026-03-10 21:00:00       102.5  102.500000  -0.244600        38.22   
1 2026-03-10 21:00:01        99.3  100.900000  -0.355694        38.42   
2 2026-03-10 21:00:02       103.2  101.666667  -0.302462        38.43   
3 2026-03-10 21:00:03       107.6  103.150000  -0.199468        38.34   
4 2026-03-10 21:00:04        98.8  102.280000  -0.259875        38.47   

   temp_smooth  temp_zsco

# Module 2
### Anomaly Detection

In [7]:
import pandas as pd
import numpy as np
import ast
import json
import os


# ── CONFIGURATION ─────────────────────────────────────────────────────────────

# Military/working/large breeds we will be evaluating in this dataset
TARGET_BREEDS = [
    "beauceron",            # French military/police dog
    "berger",               # German Shepherd family
    "tervueren",            # Belgian Malinois — primary MWD breed worldwide
    "doberman",             # Classic military/guard breed
    "husky",                # Sled/patrol dog
    "labrador",             # Detection & search-rescue dog
    "boxer",                # Historical military use
    "braque_de_weimar",     # Hunting/patrol
    "weimaraner",           # Same family as braque_de_weimar
    "australian_shepherd",  # Working/herding dog
]

DATASET_PATH = "data/dataset.csv"   # adjust if in a subfolder


# ── 1. LOAD & INITIAL INSPECT ────────────────────────────────────────────────

df_raw = pd.read_csv(DATASET_PATH)
print(f"Raw dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Breeds present:\n{df_raw['breeds'].value_counts()}\n")


# ── 2. FILTER: BREED + DATA QUALITY ──────────────────────────────────────────
"""
THREE FILTERS APPLIED:
  A) Breed filter   — keep only target military/working breeds
  B) Bad ECG filter — bad_ecg == '[]' means no corrupted signal windows
                      Corrupted signal = unreliable HR/BR readings
  C) Null filter    — drop rows missing segments_hr or segments_br
                      (can't compute features without signal data)
"""

# A: Breed filter
df = df_raw[df_raw["breeds"].isin(TARGET_BREEDS)].copy()

df = df.sort_values(by=["breeds", "pet_id"]).reset_index(drop=True)

def get_breed_code(breed):
    words = breed.split("_")
    if len(words) == 1:
        return breed[:3].upper()
    return "".join([w[0] for w in words]).upper()

df["breed_code"] = df["breeds"].apply(get_breed_code)
df["dog_seq"] = df.groupby("breeds").cumcount() + 1
df["dog_uid"] = df["breed_code"] + "_" + df["dog_seq"].astype(str).str.zfill(3)

print(f"After breed filter: {len(df)} rows")
print(df["breeds"].value_counts())

# B: Clean ECG only
df = df[df["bad_ecg"] == "[]"].copy()
print(f"\nAfter bad_ecg filter: {len(df)} rows")

# C: Drop nulls in signal columns
df = df.dropna(subset=["segments_hr", "segments_br"]).copy()
print(f"After null filter: {len(df)} rows")


# ── 3. PARSING HELPER ────────────────────────────────────────────────────────
"""
HOW ast.literal_eval WORKS:
  The column contains strings like:
    "[{'deb': 90.0, 'fin': 110.0, 'value': 56.39}, ...]"
  This is valid Python literal syntax but stored as a string.
  ast.literal_eval safely parses it back into an actual Python list of dicts.
  We then pull out the 'value' field from each dict.

  We use ast.literal_eval instead of json.loads because:
    - json.loads requires double quotes; Python dicts use single quotes
    - ast.literal_eval handles Python literals natively
"""

def parse_segments(s):
    """Parse a string-encoded list of segment dicts, return list of values."""
    try:
        segments = ast.literal_eval(s)
        return [seg["value"] for seg in segments if "value" in seg]
    except Exception:
        return []

def parse_pulses(s):
    """Parse ecg_pulses string into a list of floats."""
    try:
        return ast.literal_eval(s)
    except Exception:
        return []


# ── 4. FEATURE ENGINEERING ───────────────────────────────────────────────────
"""
FROM EACH DOG'S RECORDING WE EXTRACT:

Heart Rate features (from segments_hr):
  hr_mean  — average HR across all segments (baseline health indicator)
  hr_max   — peak HR (stress/exertion indicator)
  hr_min   — lowest HR (recovery indicator)
  hr_std   — variability of HR across segments

Breathing Rate features (from segments_br):
  br_mean, br_max, br_min, br_std — same as above for breathing rate
  High br_mean + high hr_mean = strong heat stress signal

Heart Rate Variability (HRV) from ecg_pulses:
  HRV = std of RR intervals (time between consecutive heartbeats)
  WHY THIS MATTERS:
    - Low HRV = stress, fatigue, illness (sympathetic nervous system dominant)
    - High HRV = relaxed, healthy (parasympathetic dominant)
    - HRV is used in human athletes and military personnel for stress monitoring
    - Same physiology applies to dogs
  RR interval = difference between consecutive pulse timestamps (in seconds)
"""

records = []

for _, row in df.iterrows():
    hr_vals = parse_segments(row["segments_hr"])
    br_vals = parse_segments(row["segments_br"])
    pulses  = parse_pulses(row["ecg_pulses"]) if pd.notna(row["ecg_pulses"]) else []

    # Skip rows with empty parsed values
    if not hr_vals or not br_vals:
        continue

    # HR features
    hr_mean = np.mean(hr_vals)
    hr_max  = np.max(hr_vals)
    hr_min  = np.min(hr_vals)
    hr_std  = np.std(hr_vals)

    # BR features
    br_mean = np.mean(br_vals)
    br_max  = np.max(br_vals)
    br_min  = np.min(br_vals)
    br_std  = np.std(br_vals)

    # HRV: std of RR intervals
    if len(pulses) > 1:
        rr_intervals = np.diff(pulses)          # time between beats
        hrv = np.std(rr_intervals)              # higher = healthier
        rr_mean = np.mean(rr_intervals)
    else:
        hrv = np.nan
        rr_mean = np.nan

    records.append({
        "dog_id":   row["dog_uid"],
        "breed":    row["breeds"],
        "age":      row["age"],
        "weight":   row["weight"],
        "duration": row["duration"],
        # HR features
        "hr_mean":  round(hr_mean, 2),
        "hr_max":   round(hr_max,  2),
        "hr_min":   round(hr_min,  2),
        "hr_std":   round(hr_std,  2),
        # BR features
        "br_mean":  round(br_mean, 2),
        "br_max":   round(br_max,  2),
        "br_min":   round(br_min,  2),
        "br_std":   round(br_std,  2),
        # HRV
        "hrv":      round(hrv,     4) if not np.isnan(hrv) else None,
        "rr_mean":  round(rr_mean, 4) if not np.isnan(rr_mean) else None,
    })

feat_df = pd.DataFrame(records)
print(f"\nFeature matrix: {feat_df.shape}")
print(feat_df.describe())


# ── 5. ANOMALY DETECTION (ISOLATION FOREST) ──────────────────────────────────
"""
WHY NO SUPERVISED MODEL HERE:
  Unlike Module 1 (where we injected anomalies and knew exactly which rows
  were anomalous), this real dataset has NO anomaly labels.
  A vet would need to label readings as "stressed/sick" vs "normal" for
  supervised training. That's exactly the kind of labeled data you'd
  collect in a real MWD deployment.

  Isolation Forest is perfect for this scenario — it finds outliers
  purely from the feature distribution, no labels needed.

FEATURES USED:
  We use hr_mean, hr_max, hr_std, br_mean, br_std, hrv.
  We drop rows where hrv is null (no pulse data) for the ML step.
"""

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

ML_FEATURES = ["hr_mean", "hr_max", "hr_std", "br_mean", "br_std"]

ml_df = feat_df.dropna(subset=ML_FEATURES).copy()

# Use HRV only if sufficient coverage
if "hrv" in ml_df.columns:
    hrv_available = ml_df["hrv"].notna().sum()
    print(f"\nHRV available for {hrv_available}/{len(ml_df)} rows")

    if hrv_available > 0.5 * len(ml_df):
        ML_FEATURES = ML_FEATURES + ["hrv"]
        ml_df = ml_df.dropna(subset=["hrv"])
        print("Using HRV in model")
    else:
        print("Skipping HRV due to insufficient data")

print(f"Using features: {ML_FEATURES}")
print(f"Rows for anomaly detection: {len(ml_df)}")

X = ml_df[ML_FEATURES].values

# Standardize — Isolation Forest works better on scaled data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
preds = iso.fit_predict(X_scaled)
ml_df["anomaly"] = (preds == -1).astype(int)

print(f"\nIsolation Forest flagged {ml_df['anomaly'].sum()} anomalies "
      f"out of {len(ml_df)} recordings")

print("\nAnomaly breakdown by breed:")
print(ml_df.groupby("breed")["anomaly"].agg(["sum", "count"])
      .rename(columns={"sum": "flagged", "count": "total"}))

print("\nSample anomalous recordings:")
anomalous = ml_df[ml_df["anomaly"] == 1][
    ["breed", "age", "weight", "hr_mean", "hr_max", "br_mean", "hrv"]
].head(5)
print(anomalous)


# ── 6. BUILD EVENT OBJECTS ───────────────────────────────────────────────────

HR_HIGH = feat_df["hr_mean"].mean() + 2 * feat_df["hr_mean"].std()
HR_LOW  = feat_df["hr_mean"].mean() - 2 * feat_df["hr_mean"].std()

HRV_LOW = feat_df["hrv"].mean() - 1.5 * feat_df["hrv"].std()
BR_HIGH = feat_df["br_mean"].mean() + 2 * feat_df["br_mean"].std()

print("\nDynamic thresholds:")
print("HR_HIGH:", HR_HIGH)
print("HR_LOW:", HR_LOW)
print("HRV_LOW:", HRV_LOW)
print("BR_HIGH:", BR_HIGH)

def build_real_event(row):
    flags = []

    # --- Absolute thresholds ---
    if row["hr_mean"] > HR_HIGH:
        flags.append("high_hr")
    elif row["hr_mean"] < HR_LOW:
        flags.append("low_hr")

    if pd.notna(row.get("hrv")) and row["hrv"] < HRV_LOW:
        flags.append("low_hrv")

    if row["br_mean"] > BR_HIGH:
        flags.append("high_br")

    # --- RELATIVE anomaly flags (THIS FIXES YOUR ISSUE) ---
    if row["anomaly"] == 1:
        flags.append("physiological_anomaly")

        # compare to dataset mean
        if row["hr_mean"] < feat_df["hr_mean"].mean():
            flags.append("below_normal_hr")

        if row["hr_mean"] > feat_df["hr_mean"].mean():
            flags.append("above_normal_hr")

        if row["hrv"] < feat_df["hrv"].mean():
            flags.append("reduced_hrv")

        if row["br_mean"] > feat_df["br_mean"].mean():
            flags.append("elevated_br")

    return {
        "dog_id": str(row["dog_id"]),
        "breed": row["breed"],
        "age": row["age"],
        "weight": row["weight"],
        "heart_rate": row["hr_mean"],
        "hr_max": row["hr_max"],
        "breathing_rate": row["br_mean"],
        "hrv": row.get("hrv"),
        "flags": flags,
    }

events = [build_real_event(r) for _, r in ml_df[ml_df["anomaly"] == 1].iterrows()]
print(f"\nBuilt {len(events)} event objects")
if events:
    print("\nSample event:")
    print(json.dumps(events[0], indent=2))


# ── SAVE ─────────────────────────────────────────────────────────────────────

os.makedirs("data", exist_ok=True)
feat_df.to_csv("data/real_sensor_features.csv", index=False)
with open("data/real_anomaly_events.json", "w") as f:
    json.dump(events, f, indent=2)

print("\nSaved:")
print("  data/real_sensor_features.csv  → feeds Module 3 (NLG)")
print("  data/real_anomaly_events.json  → feeds Module 3 (NLG)")
print("\nTop anomaly feature stats:")
print(ml_df[ml_df["anomaly"] == 1][ML_FEATURES].describe())

Raw dataset: 1123 rows, 11 columns
Breeds present:
breeds
unknown                     117
labrador                    113
dalmatien                   101
beauceron                    77
epagneul                     77
australian_shepherd          74
berger                       74
beagle                       73
fox_terrier                  42
tervueren                    40
boxer                        39
setter                       38
basset_fauve_de_bretagne     38
braque_de_weimar             37
husky                        37
cocker_anglais               37
doberman                     29
leonberg                     25
golden_retriever             13
samoyede                     12
weimaraner                   10
dogo_argentino                6
whippet                       6
berger_blanc_suisse           5
staffie                       3
Name: count, dtype: int64

After breed filter: 530 rows
breeds
labrador               113
beauceron               77
australian_shepherd     7

# Module 3
### Natural Language Generation

In [8]:
import json
import pandas as pd

# ── LOAD EVENTS FROM MODULE 2 ────────────────────────────────────────────────

with open("data/real_anomaly_events.json", "r") as f:
    events = json.load(f)

print(f"Loaded {len(events)} events")


# ── FLAG → MEANING MAPPING ───────────────────────────────────────────────────

FLAG_EXPLANATIONS = {
    "high_hr": "elevated heart rate",
    "low_hr": "below-normal heart rate",
    "low_hrv": "reduced heart rate variability",
    "very_low_hrv": "very low heart rate variability",
    "high_br": "elevated breathing rate",

    "below_normal_hr": "lower than typical heart rate",
    "above_normal_hr": "higher than typical heart rate",
    "reduced_hrv": "reduced heart rate variability",
    "elevated_br": "higher than normal breathing",

    "stress_pattern": "a stress-related physiological pattern",
    "heat_stress_risk": "possible heat stress",

    "physiological_anomaly": "an unusual physiological pattern"
}


# ── FLAG COMBINATION REASONING ───────────────────────────────────────────────

def interpret_flags(flags):
    """
    Converts raw flags into a meaningful interpretation
    """
    # Priority conditions (most important first)
    if "heat_stress_risk" in flags:
        return "possible heat stress"

    if "stress_pattern" in flags:
        return "possible stress or exertion"

    if "low_hr" in flags:
        return "possible fatigue or recovery state"

    if "high_hr" in flags:
        return "possible exertion or stress"

    if "physiological_anomaly" in flags:
        return "an unusual physiological condition"

    return "normal condition"



# ── SEVERITY CLASSIFIER ──────────────────────────────────────────────────────

def classify_severity(flags, hr, br, hrv):
    score = 0

    if "high_hr" in flags or "above_normal_hr" in flags:
        score += 2

    if "low_hr" in flags or "below_normal_hr" in flags:
        score += 1

    if "low_hrv" in flags or "reduced_hrv" in flags:
        score += 2

    if "high_br" in flags or "elevated_br" in flags:
        score += 2

    if "physiological_anomaly" in flags:
        score += 1

    # Hard overrides (important)
    if hr < 40:
        return "CRITICAL"
    if br > 30:
        return "CRITICAL"

    if score >= 5:
        return "CRITICAL"
    elif score >= 3:
        return "WARNING"
    else:
        return "INFO"
    
# ── INTERPRETATION LAYER ─────────────────────────────────────────────────────

def interpret_condition(flags):
    if "below_normal_hr" in flags and "elevated_br" in flags:
        return "Possible fatigue or physiological stress"

    if "above_normal_hr" in flags and "reduced_hrv" in flags:
        return "Possible exertion or stress"

    return "Unusual physiological condition"


# ── MAIN NLG FUNCTION ────────────────────────────────────────────────────────

def generate_alert(event):
    dog = event["dog_id"]
    hr = event["heart_rate"]
    br = event["breathing_rate"]
    hrv = event.get("hrv")
    flags = event["flags"]

    severity = classify_severity(flags, hr, br, hrv)

    # --- condition text ---
    conditions = []

    if "below_normal_hr" in flags:
        conditions.append("reduced heart rate")

    if "elevated_br" in flags:
        conditions.append("elevated breathing")

    condition_text = ", ".join(conditions) if conditions else "abnormal vitals"

    # --- interpretation (🔥 MOVE HERE) ---
    interpretation = interpret_condition(flags)

    # --- severity formatting ---
    if severity == "CRITICAL":
        prefix = "CRITICAL"
        action = "Immediate attention required"
    elif severity == "WARNING":
        prefix = "WARNING"
        action = "Check immediately"
    else:
        prefix = "INFO"
        action = "Monitor"

    alert = (
        f"{prefix}: {dog} — {condition_text}.\n"
        f"{interpretation}.\n"
        f"HR {hr:.1f} bpm, BR {br:.1f}.\n{action}."
    )

    return severity, alert


# ── GENERATE ALERTS ──────────────────────────────────────────────────────────

alerts = []

for event in events:
    severity, alert = generate_alert(event)

    alerts.append({
        "dog_id": event["dog_id"],
        "severity": severity,
        "alert": alert
    })


# ── DISPLAY SAMPLE OUTPUT ────────────────────────────────────────────────────

print("\nSample Alerts:\n")
for a in alerts[:5]:
    print(a["alert"])
    print()


# ── SAVE ─────────────────────────────────────────────────────────────────────

import os
os.makedirs("outputs", exist_ok=True)

with open("outputs/generated_alerts.json", "w") as f:
    json.dump(alerts, f, indent=2)

print("Saved → outputs/generated_alerts.json")

Loaded 28 events

Sample Alerts:

CRITICAL: AS_005 — elevated breathing.
Possible exertion or stress.
HR 57.5 bpm, BR 29.1.
Immediate attention required.

CRITICAL: AS_007 — reduced heart rate, elevated breathing.
Possible fatigue or physiological stress.
HR 55.8 bpm, BR 28.6.
Immediate attention required.

Possible fatigue or physiological stress.
HR 55.2 bpm, BR 20.6.
Check immediately.

CRITICAL: AS_010 — elevated breathing.
Possible exertion or stress.
HR 59.5 bpm, BR 32.7.
Immediate attention required.

CRITICAL: AS_062 — elevated breathing.
Possible exertion or stress.
HR 65.3 bpm, BR 15.7.
Immediate attention required.

Saved → outputs/generated_alerts.json


# Module 4
### Voice Integration

In [16]:
import whisper
import json
import re

import sounddevice as sd
from scipy.io.wavfile import write

import os
os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""

import asyncio
from gtts import gTTS
import subprocess

import nest_asyncio
nest_asyncio.apply()

with open("outputs/generated_alerts.json", "r") as f:
    alerts = json.load(f)

alert_map = {a["dog_id"]: a for a in alerts if "dog_id" in a}

# ── LOAD WHISPER MODEL ─────────────────────────────────────────────

print("Loading Whisper model...")
model = whisper.load_model("base")


# ── TEXT TO SPEECH SETUP ───────────────────────────────────────────

if os.path.exists("response.mp3"):
    os.remove("response.mp3")


def speak(text):
    tts = gTTS(text)
    tts.save("response.mp3")
    subprocess.run(["start", "response.mp3"], shell=True)


# ── LOAD ALERT DATA (Module 3 Output) ──────────────────────────────

with open("outputs/generated_alerts.json", "r") as f:
    alerts = json.load(f)


# ── SPEECH → TEXT ─────────────────────────────────────────────────

def transcribe_audio(file_path):
    result = model.transcribe(file_path)
    return result["text"]


# ── EXTRACT DOG ID ─────────────────────────────────────────────────

def extract_dog_id(text):
    text = text.upper()

    match = re.search(r"([A-Z]{2,3})[_\s-]?(\d{3})", text)

    if match:
        return f"{match.group(1)}_{match.group(2)}"

    return None


# ── QUERY HANDLER ──────────────────────────────────────────────────

def handle_query(audio_file, alert_map):
    text = transcribe_audio(audio_file)
    print("\nUser said:", text)

    dog_id = extract_dog_id(text)

    if not dog_id:
        response = "I couldn't identify the dog ID. Please try again."
        speak(response)
        return response

    dog_id = dog_id.upper()

    data = alert_map.get(dog_id)

    if not data:
        response = f"No data available for {dog_id}."
        speak(response)
        return response

    severity = data.get("severity", "UNKNOWN")
    message = data.get("message", "Keep monitoring and check vitals soon.")

    response = f"{dog_id} is currently in {severity} condition. {message}"

    speak(response)
    return response


# ── RECORD AUDIO FROM MICROPHONE ───────────────────────────────────

def record_audio(filename="query.wav", duration=5, fs=16000):
    print("\n Listening...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()

    os.makedirs("audio", exist_ok=True)
    filepath = f"audio/{filename}"

    write(filepath, fs, recording)
    print("Recording saved:", filepath)

    return filepath


# ── LIVE VOICE LOOP ────────────────────────────────────────────────

def live_voice_loop():
    print("\n=== Voice Assistant Started ===")
    print("Say something like: 'How is BEA 047 doing?'")
    print("Press Ctrl+C to stop.\n")

    while True:
        try:
            audio_file = record_audio()
            handle_query(audio_file)

        except KeyboardInterrupt:
            print("\nStopped.")
            break

        except Exception as e:
            print("Error:", e)
            
handle_query("audio/testspeech.mp3", alert_map)

Loading Whisper model...


c:\Users\Abhinav\Documents\Coding\NLP Project\venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



User said:  Hey, how is BEA-022 doing today?


'BEA_022 is currently in WARNING condition. Keep monitoring and check vitals soon.'

# Module 5
### FastAPI Integration (To be run from app.py)

In [3]:
from fastapi import FastAPI
import json

app = FastAPI()

# ── LOAD ALERTS ─────────────────────────────────────────────
with open("outputs/generated_alerts.json", "r") as f:
    alerts = json.load(f)

alert_map = {a["dog_id"]: a for a in alerts}


# ── ENDPOINT: GET ALL ALERTS ─────────────────────────────────
@app.get("/alerts")
def get_alerts():
    return alerts


# ── ENDPOINT: GET SINGLE DOG ────────────────────────────────
@app.get("/dog/{dog_id}")
def get_dog(dog_id: str):
    dog_id = dog_id.upper()

    if dog_id not in alert_map:
        return {"error": "Dog not found"}

    return alert_map[dog_id]


# ── ENDPOINT: CRITICAL DOGS ─────────────────────────────────
@app.get("/critical")
def get_critical():
    critical = [a for a in alerts if a["severity"] == "CRITICAL"]
    return critical


# ── ENDPOINT: WARNING DOGS ─────────────────────────────────
@app.get("/warning")
def get_warning():
    warning = [a for a in alerts if a["severity"] == "WARNING"]
    return warning


# ── SMART QUERY (LIKE YOUR TEXT SYSTEM) ─────────────────────
@app.get("/query")
def query(q: str):
    q = q.lower()

    if "critical" in q:
        return [a["dog_id"] for a in alerts if a["severity"] == "CRITICAL"]

    if "warning" in q:
        return [a["dog_id"] for a in alerts if a["severity"] == "WARNING"]

    # fallback → extract dog id
    import re
    match = re.search(r"([A-Z]{2,3})[_\s-]?(\d{3})", q.upper())

    if match:
        dog_id = f"{match.group(1)}_{match.group(2)}"
        return alert_map.get(dog_id, {"error": "Dog not found"})

    return {"error": "Invalid query"}

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Ground truth (actual labels)
y_true = ["normal", "high_hr", "high_temp", "normal", "high_hr"]

# Model predictions
y_pred = ["normal", "high_hr", "normal", "normal", "high_hr"]

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Precision, Recall, F1 (macro = balanced across classes)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

# Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred)

# Detailed Report
report = classification_report(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", report)

Accuracy: 0.8
Precision: 0.5555555555555555
Recall: 0.6666666666666666
F1 Score: 0.6

Confusion Matrix:
 [[2 0 0]
 [0 0 1]
 [0 0 2]]

Classification Report:
               precision    recall  f1-score   support

     high_hr       1.00      1.00      1.00         2
   high_temp       0.00      0.00      0.00         1
      normal       0.67      1.00      0.80         2

    accuracy                           0.80         5
   macro avg       0.56      0.67      0.60         5
weighted avg       0.67      0.80      0.72         5



c:\Users\Abhinav\Documents\Coding\NLP Project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Abhinav\Documents\Coding\NLP Project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Abhinav\Documents\Coding\NLP Project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi